In [59]:
import pandas as pd
import numpy as np
from pathlib import Path 

## Stock data

In [61]:
stock_dir = Path("/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/stocks_daily/")

files = sorted(stock_dir.glob("stocks_daily_*.parquet"))
print("Found files:", [f.name for f in files])

Found files: ['stocks_daily_1990_1994.parquet', 'stocks_daily_1995_1999.parquet', 'stocks_daily_2000_2002.parquet', 'stocks_daily_2003_2005.parquet', 'stocks_daily_2006_2008.parquet', 'stocks_daily_2009_2011.parquet', 'stocks_daily_2012_2014.parquet', 'stocks_daily_2015_2017.parquet', 'stocks_daily_2018_2019.parquet', 'stocks_daily_2020_2021.parquet', 'stocks_daily_2022_2023.parquet', 'stocks_daily_2024_2024.parquet']


In [62]:
df_list = []
for f in files:
    df = pd.read_parquet(f)

    # keep only useful columns
    keep = ["PERMNO", "date", "PRC", "RET", "VOL",
            "SHROUT", "BIDLO", "ASKHI", "NUMTRD"]
    df = df[[c for c in keep if c in df.columns]]

    # date -> datetime
    df["date"] = pd.to_datetime(df["date"].astype(str))

    # basic features
    df["log_ret"] = np.log1p(df["RET"].fillna(0))  # for vol
    df["dollar_volume"] = df["PRC"].abs() * df["VOL"]
    df["mktcap"] = df["PRC"].abs() * df["SHROUT"]
    df["turnover"] = df["VOL"] / df["SHROUT"]
    df["mid"] = (df["BIDLO"] + df["ASKHI"]) / 2
    df["bidask"] = (df["ASKHI"] - df["BIDLO"]) / df["mid"]

    df_list.append(df)

stocks_daily = pd.concat(df_list, ignore_index=True)

# ---- 2. Monthly aggregation (final stock features) ----
monthly_stock = (
    stocks_daily.set_index("date")
    .groupby("PERMNO")
    .resample("M")
    .agg({
        "RET":      lambda x: (1 + x.fillna(0)).prod() - 1,  # monthly return
        "log_ret":  "std",                                   # monthly vol
        "dollar_volume": "sum",
        "turnover": "mean",
        "mktcap": "last",
        "bidask": "mean",
        "NUMTRD": "sum",
        "PRC": "mean",
    })
    .reset_index()
)

monthly_stock = monthly_stock.rename(columns={
    "RET": "ret",
    "log_ret": "vol",
    "dollar_volume": "dvol",
    "turnover": "turnover",
    "mktcap": "mktcap",
    "bidask": "bidask",
    "NUMTRD": "numtrades",
    "PRC": "price_mean",
})

print(monthly_stock.head())

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_6806/4277642577.py:29: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  .resample("M")


   PERMNO       date           ret       vol        dvol  turnover     mktcap  \
0   10001 1990-01-31 -1.851994e-02  0.010803  351521.875  1.568004  10156.125   
1   10001 1990-02-28 -6.289601e-03  0.014507  147554.500  0.765372  10092.250   
2   10001 1990-03-31  1.282060e-02  0.024371  126388.125  0.566047  10141.625   
3   10001 1990-04-30 -2.384186e-07  0.014212  165962.500  0.810370  10141.625   
4   10001 1990-05-31 -1.265705e-02  0.017681  274941.250  1.234177  10013.250   

     bidask  numtrades  price_mean  
0  0.006247       59.0    8.204545  
1  0.008640       28.0    5.743421  
2  0.018875       35.0    5.460227  
3  0.017085       29.0    1.018750  
4  0.014960       41.0    5.386364  


## Fundamental Data

In [63]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/"
fundamental_data = pd.read_parquet(dir + "fundamentals_quarterly.parquet")
fundamental_data.head()

,GVKEY,LPERMNO,datadate,fyearq,fqtr,indfmt,consol,popsrc,datafmt,cusip,...,ceqq,cshoq,ltq,niq,oibdpq,revtq,xintq,costat,prccq,gsector
0,1004,54594,19900228,1989,3,INDL,C,D,STD,000361105,...,186.173,16.070,197.618,6.109,14.251,112.278,2.758,A,31.124977,20.0
1,1004,54594,19900531,1989,4,INDL,C,D,STD,000361105,...,189.548,16.082,198.973,6.224,13.137,119.396,2.309,A,21.249998,20.0
2,1004,54594,19900831,1990,1,INDL,C,D,STD,000361105,...,194.018,16.086,191.734,6.697,15.400,116.092,2.607,A,15.874998,20.0
3,1004,54594,19901130,1990,2,INDL,C,D,STD,000361105,...,189.503,15.879,193.475,0.126,8.871,115.808,2.708,A,11.874999,20.0
4,1004,54594,19910228,1990,3,INDL,C,D,STD,000361105,...,191.761,15.891,191.593,3.977,11.344,117.820,2.587,A,12.874998,20.0


In [69]:
# Keep only the columns we need
cols_keep = [
    "GVKEY", "LPERMNO", "datadate",
    "atq", "ceqq", "cshoq", "ltq", "niq",
    "oibdpq", "revtq", "xintq", "prccq", "cusip",
    "curcdq", "consol", "indfmt", "datafmt", "costat", "gsector"
]
fund = fundamental_data[cols_keep].copy()

# Parse date and sort
fund["datadate"] = pd.to_datetime(fund["datadate"])
fund = fund.sort_values(["LPERMNO", "datadate"])
fund = fund.drop_duplicates(subset=["LPERMNO", "datadate"])


# Create quarterly features

# Avoid division by zero
eps = 1e-9

# Firm size
fund["log_atq"] = np.log(fund["atq"] + eps)

# Leverage ratios
fund["lev_total"] = fund["ltq"] / (fund["atq"] + eps)      # liabilities / assets
fund["equity_ratio"] = fund["ceqq"] / (fund["atq"] + eps)  # equity / assets

# Profitability
fund["roa"] = fund["niq"] / (fund["atq"] + eps)            # net income / assets
fund["profit_margin"] = fund["niq"] / (fund["revtq"] + eps)  # net income / revenue

# Interest coverage
fund["int_coverage"] = fund["oibdpq"] / (fund["xintq"].replace(0, np.nan))

# Market-based fundamentals (need price * shares)
fund["mkt_cap"] = fund["prccq"] * fund["cshoq"]
fund["market_to_book"] = fund["mkt_cap"] / (fund["ceqq"] + eps)

# Growth variables (quarter-over-quarter, within firm)
fund[["atq_growth", "revtq_growth", "niq_growth"]] = (
    fund.groupby("LPERMNO")[["atq", "revtq", "niq"]]
        .pct_change()
)

# Convert quarterly to monthly (forward-fill within firm)

def to_monthly(group: pd.DataFrame) -> pd.DataFrame:
    """
    For each firm, set datadate as index, resample to month-end,
    and forward-fill fundamentals and features.
    """
    g = group.set_index("datadate")
    # Resample to calendar month-end
    g_monthly = g.resample("M").ffill()
    return g_monthly

fund_monthly = (
    fund
    .groupby("LPERMNO", group_keys=False)
    .apply(to_monthly)
    .reset_index()  # datadate becomes month-end date
)

print(fund_monthly.head())

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_6806/2142044603.py:42: FutureWarning: The default fill_method='ffill' in DataFrameGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .pct_change()
/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_6806/2142044603.py:54: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  g_monthly = g.resample("M").ffill()


    datadate  GVKEY  LPERMNO       atq    ceqq   cshoq       ltq    niq  \
0 1970-01-31  12994    10001   183.840  94.352  10.520    89.488 -0.386   
1 1970-01-31  19049    10002  1954.522  44.277  17.733  1861.047 -9.502   
2 1970-01-31  16950    10003   320.338  16.114   5.043   304.224  0.008   
3 1970-01-31  12087    10005     0.692   0.660   8.340     0.032 -0.036   
4 1970-01-31   9636    10007     3.507  -1.292   4.134     4.742 -1.795   

   oibdpq   revtq  ...  lev_total  equity_ratio       roa profit_margin  \
0   1.191  16.758  ...   0.486771      0.513229 -0.002100     -0.023034   
1  -0.371     NaN  ...   0.952175      0.022654 -0.004862           NaN   
2   1.542     NaN  ...   0.949697      0.050303  0.000025           NaN   
3  -0.012   0.031  ...   0.046243      0.953757 -0.052023     -1.161290   
4     NaN   0.051  ...   1.352153     -0.368406 -0.511833    -35.196078   

  int_coverage     mkt_cap market_to_book atq_growth  revtq_growth  niq_growth  
0     1.616011  1

/var/folders/b4/rh5hpt517nb6_mc9jyr1b1440000gn/T/ipykernel_6806/2142044603.py:60: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(to_monthly)


## Merge monthly_stock + fund_monthly

In [70]:
# align keys (LPERMNO -> PERMNO, datadate -> date)
fund_monthly = fund_monthly.rename(
    columns={"LPERMNO": "PERMNO", "datadate": "date"}
)

# make sure date columns are datetime
fund_monthly["date"] = pd.to_datetime(fund_monthly["date"])
monthly_stock["date"] = pd.to_datetime(monthly_stock["date"])

# merge on PERMNO + date
merged_1 = monthly_stock.merge(
    fund_monthly,
    on=["PERMNO", "date"],
    how="left"
)

print(merged_1.head())
print(merged_1.shape)

   PERMNO       date           ret       vol        dvol  turnover     mktcap  \
0   10001 1990-01-31 -1.851994e-02  0.010803  351521.875  1.568004  10156.125   
1   10001 1990-02-28 -6.289601e-03  0.014507  147554.500  0.765372  10092.250   
2   10001 1990-03-31  1.282060e-02  0.024371  126388.125  0.566047  10141.625   
3   10001 1990-04-30 -2.384186e-07  0.014212  165962.500  0.810370  10141.625   
4   10001 1990-05-31 -1.265705e-02  0.017681  274941.250  1.234177  10013.250   

     bidask  numtrades  price_mean  ...  lev_total  equity_ratio  roa  \
0  0.006247       59.0    8.204545  ...        NaN           NaN  NaN   
1  0.008640       28.0    5.743421  ...        NaN           NaN  NaN   
2  0.018875       35.0    5.460227  ...        NaN           NaN  NaN   
3  0.017085       29.0    1.018750  ...        NaN           NaN  NaN   
4  0.014960       41.0    5.386364  ...        NaN           NaN  NaN   

   profit_margin  int_coverage  mkt_cap  market_to_book  atq_growth  \
0  

## Merge macro data

In [71]:
macro = pd.read_excel(dir + "macro_data.xlsx")
# convert date to datetime
macro["date"] = pd.to_datetime(macro["date"].astype(str))

# sort for safety
macro = macro.sort_values("date")

# ensure merged (stock+fund) also uses datetime for date
merged_1["date"] = pd.to_datetime(merged_1["date"])

# merge macro into merged dataset
merged_2 = merged_1.merge(
    macro,
    on="date",
    how="left"
)

print(merged_2.head())
print("shape:", merged_2.shape)

   PERMNO       date           ret       vol        dvol  turnover     mktcap  \
0   10001 1990-01-31 -1.851994e-02  0.010803  351521.875  1.568004  10156.125   
1   10001 1990-02-28 -6.289601e-03  0.014507  147554.500  0.765372  10092.250   
2   10001 1990-03-31  1.282060e-02  0.024371  126388.125  0.566047  10141.625   
3   10001 1990-04-30 -2.384186e-07  0.014212  165962.500  0.810370  10141.625   
4   10001 1990-05-31 -1.265705e-02  0.017681  274941.250  1.234177  10013.250   

     bidask  numtrades  price_mean  ...  market_to_book  atq_growth  \
0  0.006247       59.0    8.204545  ...             NaN         NaN   
1  0.008640       28.0    5.743421  ...             NaN         NaN   
2  0.018875       35.0    5.460227  ...             NaN         NaN   
3  0.017085       29.0    1.018750  ...             NaN         NaN   
4  0.014960       41.0    5.386364  ...             NaN         NaN   

   revtq_growth  niq_growth  sp500  ir3m  ir10y  vix  gdp  cpi  
0           NaN      

In [72]:
industry = pd.read_excel(dir + "industry_data.xlsx")

# convert date column
industry["date"] = pd.to_datetime(industry["date"].astype(str))

# sector mapping
sector_map = {
    10: "XLE",
    15: "XLB",
    20: "XLI",
    25: "XLY",
    30: "XLP",
    35: "XLV",
    40: "XLF",
    45: "XLK",
    50: "XLC",
    55: "XLU",
    60: "XLRE",
}

# add ETF ticker to merged via gsector
merged_2["sector_etf"] = merged_2["gsector"].map(sector_map)

# merge ETF monthly data on (sector_etf, date)
industry = industry.rename(columns={"sector": "sector_etf"})

merged_3 = merged_2.merge(
    industry,
    on=["sector_etf", "date"],
    how="left"
)

print(merged_3.head())
print("shape:", merged_3.shape)

   PERMNO       date           ret       vol        dvol  turnover     mktcap  \
0   10001 1990-01-31 -1.851994e-02  0.010803  351521.875  1.568004  10156.125   
1   10001 1990-02-28 -6.289601e-03  0.014507  147554.500  0.765372  10092.250   
2   10001 1990-03-31  1.282060e-02  0.024371  126388.125  0.566047  10141.625   
3   10001 1990-04-30 -2.384186e-07  0.014212  165962.500  0.810370  10141.625   
4   10001 1990-05-31 -1.265705e-02  0.017681  274941.250  1.234177  10013.250   

     bidask  numtrades  price_mean  ...  niq_growth  sp500  ir3m  ir10y  vix  \
0  0.006247       59.0    8.204545  ...         NaN    NaN   NaN    NaN  NaN   
1  0.008640       28.0    5.743421  ...         NaN    NaN   NaN    NaN  NaN   
2  0.018875       35.0    5.460227  ...         NaN    NaN   NaN    NaN  NaN   
3  0.017085       29.0    1.018750  ...         NaN    NaN   NaN    NaN  NaN   
4  0.014960       41.0    5.386364  ...         NaN    NaN   NaN    NaN  NaN   

   gdp  cpi  sector_etf  price  

## Merge bond data

In [ ]:
bond_data = pd.read_parquet(dir + "bond_data_processed.parquet")
print(bond_data.head())

        date      cusip company_symbol   tmt   coupon  t_spread    yield  \
0 2002-07-31  000325AA8           AAFM  0.55  0.08875       NaN      NaN   
1 2002-08-31  000325AA8           AAFM  0.47  0.08875    0.0042  0.07731   
2 2002-09-30  000325AA8           AAFM  0.38  0.08875    0.0078  0.07933   
3 2002-10-31  000325AA8           AAFM  0.30  0.08875       NaN  0.07708   
4 2002-11-30  000325AA8           AAFM  0.21  0.08875    0.0025  0.04793   

    ret_eom  rating_A  rating_AA  ...  rating_BB  rating_BBB  rating_C  \
0       NaN       NaN        NaN  ...        NaN         NaN       NaN   
1  0.005162       0.0        0.0  ...        0.0         0.0       0.0   
2  0.007239       0.0        0.0  ...        0.0         0.0       0.0   
3  0.014820       0.0        0.0  ...        0.0         0.0       0.0   
4 -0.000199       0.0        0.0  ...        0.0         0.0       0.0   

   rating_CC  rating_CCC  rating_D  upgrade  downgrade    gs3m  term_spread  
0        NaN        

In [75]:
# helper: normalize CUSIP to 8-char root
def normalize_cusip(series):
    """Convert CUSIP-like codes to 8-char issuer-level root."""
    s = series.astype(str).str.strip().str.upper()
    # drop possible ".0" from float-like strings
    s = s.str.replace(r"\.0$", "", regex=True)
    # pad with zeros on the left if too short
    s = s.str.zfill(8)
    # use first 8 chars (ignore 9th check digit if exists)
    return s.str.slice(0, 8)

# ensure date is datetime
bond_data["date"] = pd.to_datetime(bond_data["date"])

# create 8-char cusip key
bond_data["cusip8"] = normalize_cusip(bond_data["cusip"])


merged_3["date"] = pd.to_datetime(merged_3["date"])
merged_3["cusip8"] = normalize_cusip(merged_3["cusip"])

# merge: left join on bond_data
merged_all = bond_data.merge(
    merged_3,
    on=["cusip8", "date"],
    how="left",
    suffixes=("", "_firm")  # avoid name collision
)

print(merged_all.head())
print(merged_all.shape)

        date      cusip company_symbol   tmt   coupon  t_spread    yield  \
0 2002-07-31  000325AA8           AAFM  0.55  0.08875       NaN      NaN   
1 2002-08-31  000325AA8           AAFM  0.47  0.08875    0.0042  0.07731   
2 2002-09-30  000325AA8           AAFM  0.38  0.08875    0.0078  0.07933   
3 2002-10-31  000325AA8           AAFM  0.30  0.08875       NaN  0.07708   
4 2002-11-30  000325AA8           AAFM  0.21  0.08875    0.0025  0.04793   

    ret_eom  rating_A  rating_AA  ...  niq_growth  sp500  ir3m  ir10y  vix  \
0       NaN       NaN        NaN  ...         NaN    NaN   NaN    NaN  NaN   
1  0.005162       0.0        0.0  ...         NaN    NaN   NaN    NaN  NaN   
2  0.007239       0.0        0.0  ...         NaN    NaN   NaN    NaN  NaN   
3  0.014820       0.0        0.0  ...         NaN    NaN   NaN    NaN  NaN   
4 -0.000199       0.0        0.0  ...         NaN    NaN   NaN    NaN  NaN   

   gdp  cpi  sector_etf  price  return  
0  NaN  NaN         NaN    NaN   

In [76]:
dir = "/Users/yutung/MQF/Machine learning/Project/ml_project/data/processed/"
merged_all.to_parquet(dir + "merged_all.parquet", index=False)